# Electrolyte generalization benchmark v0.5: example usage

This notebook shows how to (1) inspect the curated data and its provenance, (2) read the frozen evaluation design,
(3) evaluate **your own model** on a frozen task with the benchmark's metrics, and (4) trace an error back to the raw record.

Run `uv run elbench all` first so the curated tables, splits and baseline results exist.

In [1]:
import os, pathlib
import numpy as np
import pandas as pd
from elbench.config import load_config
from elbench.curate import load_processed

root = pathlib.Path.cwd() if (pathlib.Path.cwd() / "configs").exists() else pathlib.Path.cwd().parent
cfg = load_config(root / "configs" / "benchmark_v0.5.yaml")
meas, comps = load_processed(cfg)
pd.set_option("display.width", 160)
meas.inclusion_status.value_counts()

inclusion_status
included    18172
excluded      688
Name: count, dtype: int64

## 1. Canonical records and provenance
One row per reported measurement. Excluded rows stay visible with a reason; every value keeps its original unit and a link to the raw record.

In [2]:
cols = ["measurement_id", "source_doi", "chemistry_system_id", "ratio_basis", "concentration", "concentration_basis",
        "temperature_K", "conductivity_mS_cm", "conductivity_original", "conductivity_unit_original", "track_id", "source_record_link"]
meas[meas.inclusion_status == "included"][cols].sample(5, random_state=0)

,measurement_id,source_doi,chemistry_system_id,ratio_basis,concentration,concentration_basis,temperature_K,conductivity_mS_cm,conductivity_original,conductivity_unit_original,track_id,source_record_link
190,calisol23:190,10.1016/j.jpowsour.2011.07.071,LiPF6|DMC+EC,mol,1.000000,molarity,348.419425,21.401949,21.401949,mS/cm,molar_x,https://github.com/Pele0599/CALiSol-23/blob/02...
9927,calisol23:9927,10.1149/1.1833611,LiBOB|PC,w,0.623529,molality,263.150000,0.903226,0.903226,mS/cm,molal_w,https://github.com/Pele0599/CALiSol-23/blob/02...
17894,rahmanian2023:4069,10.1038/s41597-023-01936-3,LiPF6|EC+EMC+PC,w,0.201524,molality,283.150000,2.831906,0.002832,S/cm,molal_w,https://zenodo.org/records/7244939/files/Condu...
5250,calisol23:5250,10.1149/1.1527939,LiBPFPB|PC,mol,2.150000,molarity,369.717882,0.355022,0.355022,mS/cm,molar_x,https://github.com/Pele0599/CALiSol-23/blob/02...
8842,calisol23:8842,10.1149/1.1833611,LiBOB|PC,w,0.475410,molality,263.150000,0.950234,0.950234,mS/cm,molal_w,https://github.com/Pele0599/CALiSol-23/blob/02...


In [3]:
meas[meas.inclusion_status == "excluded"].exclusion_reason.value_counts()

exclusion_reason
missing_conductivity                  523
unresolved_component                   96
negative_conductivity_nonphysical      27
source_quarantine                      23
confirmed_compilation_error            10
reported_zero_conductivity              8
negative_concentration_nonphysical      1
Name: count, dtype: int64[pyarrow]

In [4]:
inc = meas[meas.inclusion_status == "included"]
inc.groupby("track_id").agg(measurements=("measurement_id", "size"), formulations=("formulation_id", "nunique"),
                            source_groups=("source_group_id", "nunique"), chemistry_systems=("chemistry_system_id", "nunique"))

,measurements,formulations,source_groups,chemistry_systems
track_id,,,,
molal_w,15506,1751,8,21
molar_v,1269,212,5,88
molar_x,1397,105,9,28


## 2. Frozen evaluation design
Each fold carries an interpretation label. `combined_shift` means chemistry, source or conditions change together, so a failure cannot be attributed to one factor.

In [5]:
folds = pd.read_csv(cfg.p("splits") / "folds.csv")
folds[folds.track_id == "molal_w"][["task_id", "fold_id", "n_train", "n_test", "interpretation_label", "n_formulations", "n_source_groups"]]

,task_id,fold_id,n_train,n_test,interpretation_label,n_formulations,n_source_groups
0,interp_formulation,interp/0,8570,1901,supported_comparison,319,7
1,interp_formulation,interp/1,8339,2132,supported_comparison,324,7
2,interp_formulation,interp/2,8337,2134,supported_comparison,336,7
3,interp_formulation,interp/3,8293,2178,supported_comparison,330,7
4,interp_formulation,interp/4,8354,2117,supported_comparison,337,7
15,source_transfer,source/doi:10.1149/1.1403730,10146,100,supported_comparison,26,1
16,source_transfer,source/doi:10.1149/1.1833611,8041,292,supported_comparison,28,1
17,source_transfer,source/doi:10.1149/1.1914757,9661,100,supported_comparison,20,1
18,source_transfer,source/doi:10.1149/2.0271802jes,10143,60,supported_comparison,11,1
24,author_transfer,authors/AG_ARL_Ding_Jow,373,100,supported_comparison,26,1


## 3. Evaluate your own model
`iter_folds` yields features and training labels for each outer fold, plus nested development folds for tuning.
Test labels are never handed out; `score_predictions` joins them by ID and compares against the stored baselines with a paired bootstrap over groups.

The example model below is a random forest that tunes `min_samples_leaf` on the development folds only.

In [6]:
from sklearn.ensemble import RandomForestRegressor
from elbench.submission import iter_folds, score_predictions

TASK, TRACK = "source_transfer", "molal_w"

def fit(X, y, leaf):
    return RandomForestRegressor(n_estimators=100, min_samples_leaf=leaf, random_state=0, n_jobs=-1).fit(X, np.log10(y))

preds = []
for fold in iter_folds(cfg, TASK, TRACK):
    # choose the leaf size on development folds (other held-out sources inside the training data)
    scores = {}
    for leaf in (1, 5, 20):
        dev_err = [np.mean(np.abs(fit(d.X_train, d.y_train, leaf).predict(d.X_test) - np.log10(d.y_dev))) for d in fold.dev_folds]
        scores[leaf] = np.mean(dev_err)
    best = min(scores, key=scores.get)
    model = fit(fold.X_train, fold.y_train, best)
    preds.append(fold.prediction_frame(10 ** model.predict(fold.X_test)))
    print(f"{fold.fold_id:45s} leaf={best:2d}  n_test={len(fold.X_test)}")
submission = pd.concat(preds)

source/doi:10.1149/1.1403730                  leaf= 1  n_test=100


source/doi:10.1149/1.1833611                  leaf= 1  n_test=292


source/doi:10.1149/1.1914757                  leaf= 1  n_test=100


source/doi:10.1149/2.0271802jes               leaf= 1  n_test=60


In [7]:
result = score_predictions(cfg, TASK, TRACK, submission)
result["summary"][["n_rows", "n_groups", "group_mean_mae_log10", "sample_mae_log10", "worst_quartile_mae_log10", "invalid_rate"]]

,n_rows,n_groups,group_mean_mae_log10,sample_mae_log10,worst_quartile_mae_log10,invalid_rate
0,552,4,0.081878,0.105398,0.14452,0.0


In [8]:
# paired bootstrap over held-out sources; diff_vs_reference is baseline minus submission (positive = submission better)
result["comparison"][["model", "n_groups", "group_mean_mae_log10", "ci_low", "ci_high", "diff_vs_reference", "diff_ci_low", "diff_ci_high"]]

,model,n_groups,group_mean_mae_log10,ci_low,ci_high,diff_vs_reference,diff_ci_low,diff_ci_high
0,extra_trees,4,0.074663,0.036338,0.112988,-0.007215,-0.012749,-0.001681
1,hist_gb,4,0.089653,0.046425,0.129918,0.007774,0.001558,0.018840
2,knn,4,0.075644,0.038266,0.113022,-0.006234,-0.012357,-0.000302
3,median,4,0.430503,0.386327,0.474678,0.348624,0.286662,0.412732
4,ridge,4,0.136047,0.112575,0.159519,0.054168,-0.000412,0.096969
5,submission,4,0.081878,0.041867,0.122887,0.000000,0.000000,0.000000


## 4. Trace an error back to its raw record

In [9]:
g = result["groups"].sort_values("mae_log10", ascending=False)
worst_source = g.group.iat[0]
print("worst held-out source:", worst_source, "MAE log10 =", round(g.mae_log10.iat[0], 3))
sub = submission.merge(meas[["measurement_id", "source_group_id", "conductivity_mS_cm", "temperature_original", "concentration_original",
                             "conductivity_original", "conductivity_unit_original", "source_record_link"]], on="measurement_id")
sub = sub[sub.source_group_id == worst_source]
sub["abs_log_err"] = np.abs(np.log10(sub.prediction_mS_cm) - np.log10(sub.conductivity_mS_cm))
sub.sort_values("abs_log_err", ascending=False).head(5)

worst held-out source: doi:10.1149/1.1833611 MAE log10 = 0.145


,fold_id,measurement_id,prediction_mS_cm,status,source_group_id,conductivity_mS_cm,temperature_original,concentration_original,conductivity_original,conductivity_unit_original,source_record_link,abs_log_err
210,source/doi:10.1149/1.1833611,calisol23:8861,0.265147,ok,doi:10.1149/1.1833611,0.000585,233.75,0.593443,0.000585,mS/cm,https://github.com/Pele0599/CALiSol-23/blob/02...,2.655975
261,source/doi:10.1149/1.1833611,calisol23:8917,0.450138,ok,doi:10.1149/1.1833611,0.002342,243.65,0.983607,0.002342,mS/cm,https://github.com/Pele0599/CALiSol-23/blob/02...,2.283774
251,source/doi:10.1149/1.1833611,calisol23:8906,0.455237,ok,doi:10.1149/1.1833611,0.018735,243.65,0.891803,0.018735,mS/cm,https://github.com/Pele0599/CALiSol-23/blob/02...,1.385576
359,source/doi:10.1149/1.1833611,calisol23:9935,0.285357,ok,doi:10.1149/1.1833611,0.032258,233.75,0.717647,0.032258,mS/cm,https://github.com/Pele0599/CALiSol-23/blob/02...,0.946750
348,source/doi:10.1149/1.1833611,calisol23:9924,0.262650,ok,doi:10.1149/1.1833611,0.032258,233.75,0.623529,0.032258,mS/cm,https://github.com/Pele0599/CALiSol-23/blob/02...,0.910739


The baseline report (`reports/v0.5/report.html`) contains the full results, the source-familiarity investigation and three inspected failure cases.
The coverage audit (`reports/v0.5/coverage.html`) shows which comparisons the data can support.